## Objective:

To construct a vectorized mathematical engine that isolates the pure kinetic energy (variance) of an asset universe. We must physically strip the expected daily drift from the raw data. Failing to execute this step results in calculating the non-central second moment, which permanently intertwines directional drift with volatility, rendering the resulting risk factors mathematically invalid.

**The Physics Analogy:** 

Stripping the Ocean CurrentThe raw logbook ($R$) tracks the total distance each ship moved on a given day. However, the entire fleet is sitting in a slow-moving ocean current (the mean return, $\mu$) that pushes every ship forward regardless of its individual engine power. To measure exactly how violently the ships are moving in relation to one another (Covariance), we must mathematically delete the ocean current.

**Mathematical Execution:**

Assume we have ingested the raw return matrix $R \in \mathbb{R}^{T \times N}$.

1. Isolating the Current ($\mu$): We collapse the time dimension ($T$) to calculate the expected daily drift per asset.$$\mu = \frac{1}{T} \sum_{t=1}^{T} R_t$$

2. Mean Centering ($X$): We subtract the expected drift from every single daily observation.$$X = R - \mathbf{1}\mu^T$$

3. The Tangle Map ($\Sigma$): We execute the geometric dot product of the centered matrix against its transpose, scaled by degrees of freedom, to map the exact structural relationships between all assets.$$\Sigma = \frac{1}{T-1} X^T X$$

In [2]:
import numpy as np
from pathlib import Path

np.random.seed(19)

T = 1000
N = 5

R = np.random.rand(1000, 5)

### I. Mean centering

In [4]:
def mean_centering(R: np.ndarray) -> np.ndarray:
    mean = np.mean(R, axis=0)
    mean = np.reshape(mean, (R.shape[1], 1))
    ones = np.ones((R.shape[0], 1))
    
    X = R - ones @ mean.T
    
    return X

mean_centering(R)

array([[-0.40983954,  0.26495754, -0.24503435, -0.34344422, -0.18049926],
       [-0.42437358,  0.17568491,  0.31462148,  0.50116601,  0.12371491],
       [-0.29144989,  0.05273526,  0.05358763, -0.24749983, -0.39821998],
       ...,
       [-0.08805256,  0.45004471, -0.02703389, -0.13608282, -0.49060695],
       [-0.18446954,  0.37502856, -0.00604867,  0.42147286,  0.24062178],
       [ 0.37097429,  0.16744342, -0.37788568,  0.17148883,  0.34178172]],
      shape=(1000, 5))

In [5]:
### II. Empirical covariance matrix

In [6]:
def empirical_cov_matrix(R: np.ndarray) -> np.ndarray:
    X = mean_centering(R)
    return 1/(X.shape[0] - 1) * (X.T @ X)

empirical_cov_matrix(R)

array([[ 0.08487684, -0.00300396, -0.00488707,  0.00445524, -0.00180473],
       [-0.00300396,  0.07801029, -0.00405125, -0.00103501, -0.00250077],
       [-0.00488707, -0.00405125,  0.07920189, -0.00060817,  0.00286897],
       [ 0.00445524, -0.00103501, -0.00060817,  0.08053376,  0.0019792 ],
       [-0.00180473, -0.00250077,  0.00286897,  0.0019792 ,  0.08286195]])